# cli

> The command line: score a file or stdin, warm

In [ ]:
#| default_exp cli

`slopometer --path <file>` prints the worst-first report, `slopometer` alone reads stdin, and `--threshold` turns the density into an exit code for CI and hooks. Loading the model takes about a second and scoring takes milliseconds, so the command runs through [warmpy](https://github.com/AnswerDotAI/warmpy): the first call starts a background process that loads the model, later calls reuse it, and it exits after thirty idle minutes. The Claude Code stop hook runs this same command with the response text on stdin.

In [ ]:
#| export
import sys
from warmpy import warm_parse

warmpy's contract puts one obligation on this module: it must import fast, because the client process pays this import on every command. So the heavy import happens inside the function body, which only ever runs in the warm background process or in the cold fallback. `warm_parse` mirrors `call_parse`: the docments become the `--help`, and a direct Python call with arguments runs the body in this process, which is what the demo below does.

In [ ]:
from fastcore.test import *
from nbdev.config import get_config

In [ ]:
#| export
@warm_parse
def main(
    path:str=None, # Markdown or notebook file to score; stdin when omitted
    threshold:float=None, # Exit code 1 when density exceeds this
    json:bool=False, # Emit the result as JSON instead of the report
    min_words:int=150, # Minimum scored words; 0 disables the cutoff
):
    "Score Markdown or notebook prose against the aai reference-prose rules"
    from json import dumps
    from slopometer.score import score_path, score_text
    res = score_path(path, min_words=min_words) if path else score_text(sys.stdin.read(), min_words=min_words)
    if json:
        findings = [{k: getattr(f, k) for k in ('rule', 'tell', 'start', 'end', 'text', 'weight', 'suggestion')}
            for f in res.findings]
        if res.cells is not None:
            for row,f in zip(findings, res.findings): row['location'] = res.location(f)
        print(dumps(dict(density=res.density, worst=res.worst, words=res.words,
            too_short=res.too_short, min_words=res.min_words, findings=findings)))
    else: print(res)
    if threshold is not None and res.density is not None and res.density > threshold: return 1


In [ ]:
t2 = get_config().config_path/'samples'/'theory2.md'
test_eq(main(path=str(t2), threshold=5, min_words=0), None)

/Users/jhoward/aai-ws/slopometer/samples/theory2.md: density 4.5 (weight 16 on 353 prose words), worst 3
3|1d09| [3] triad (tell 20, forced symmetry): 'directory, same environment, same exit code'
7|066f| [3] passive: 'socket that does not answer is deleted'
7|066f| [3] passive: 'server with the wrong version is replaced'
9|f956| [3] passive: 'life is inferred'
15|3502| [3] rhetorical (tell 18, rhetorical questions): 'The test for every future decision: can the user tell, except by the clock?'
11|6894| [1] noun_cluster: 'mutates module state'


Machine consumers get `--json`: density, worst, scored word count, and every finding's fields. Results also include `too_short` and `min_words`. Below the cutoff, `too_short` is true, numeric scores are null, and findings are empty. Short inputs do not fail `--threshold` checks. Use `--min-words 0` to score short text deliberately.

In [ ]:
import json
from io import StringIO
from contextlib import redirect_stdout
from unittest.mock import patch

In [ ]:
buf = StringIO()
with redirect_stdout(buf): main(path=str(t2), json=True, min_words=0)
j = json.loads(buf.getvalue())
test_eq(set(j) >= {'density', 'worst', 'words', 'findings', 'too_short', 'min_words'}, True)
assert not j['too_short']
j['density'], j['findings'][0]

(4.5,
 {'rule': 'triad',
  'tell': 20,
  'start': 383,
  'end': 426,
  'text': 'directory, same environment, same exit code',
  'weight': 3,
  'suggestion': None})

In [ ]:
buf = StringIO()
with patch('sys.stdin', StringIO('robust widgets')), redirect_stdout(buf):
    test_eq(main(threshold=-1), None)
test_eq(buf.getvalue().strip(), 'too short to meter (2 scored words; minimum 150)')

buf = StringIO()
with patch('sys.stdin', StringIO('robust widgets')), redirect_stdout(buf):
    test_eq(main(json=True, threshold=0), None)
j = json.loads(buf.getvalue())
test_eq(j, dict(density=None, worst=None, words=2, too_short=True, min_words=150, findings=[]))

buf = StringIO()
with patch('sys.stdin', StringIO('robust widgets')), redirect_stdout(buf):
    test_eq(main(json=True, threshold=5, min_words=0), 1)
assert json.loads(buf.getvalue())['density'] > 5


An `.ipynb` path scores Markdown cells only. Notebook JSON findings add `location` with the cell ID, zero-based cell index, one-based cell line, and `cell_id:lineno|hash|` address. Finding `start` and `end` remain offsets into the concatenated Markdown, not the notebook JSON.

In [ ]:
from fastcore.nbio import new_nb, mk_cell, write_nb
from fastcore.tools import lnhash_at
from pathlib import Path
import tempfile


In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = Path(d)/'guide.ipynb'
    write_nb(new_nb([mk_cell('Robust widgets.', 'markdown', id='intro'), mk_cell('paradigm')]), p)
    buf = StringIO()
    with redirect_stdout(buf): test_eq(main(path=str(p), json=True, threshold=5, min_words=0), 1)
    j = json.loads(buf.getvalue())
    test_eq(j['words'], 2)
    test_eq(j['findings'][0]['location'], dict(cell_id='intro', cell_index=0, line=1,
        address='intro:' + lnhash_at('Robust widgets.', 1)))
    buf = StringIO()
    with redirect_stdout(buf): test_eq(main(path=str(p), json=True, threshold=0), None)
    assert json.loads(buf.getvalue())['too_short']
j['findings'][0]


{'rule': 'banned',
 'tell': None,
 'start': 0,
 'end': 6,
 'text': 'Robust',
 'weight': 10,
 'suggestion': 'strong',
 'location': {'cell_id': 'intro',
  'cell_index': 0,
  'line': 1,
  'address': 'intro:1|f030|'}}

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()